# Step 4: Grad-CAM
Same setup as notebook 03 (T4 GPU, mount Drive, clone/pull repo, `dvc pull`). Needs the RSNA labels CSV only for drawing the true boxes; if /content/rsna is gone, re-run the Kaggle download from notebook 01.

In [ ]:
!pip -q install --upgrade monai dvc

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
GITHUB_URL = 'https://github.com/pragya-25/medAI.git'
!git clone {GITHUB_URL} /content/medai 2>/dev/null || (cd /content/medai && git pull)
%cd /content/medai
!dvc pull

In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as patches
from ml.train import build_model
from ml.dataset import CXRDataset
from ml.gradcam import gradcam, overlay

dev = 'cuda'
model = build_model().to(dev)
model.load_state_dict(torch.load('/content/drive/MyDrive/medai_ckpt/best.pt', map_location=dev))
ds = CXRDataset('data/splits/test.csv', 'data/processed')
boxes = pd.read_csv('/content/rsna/stage_2_train_labels.csv')

In [ ]:
# Show true pneumonia cases: image | heatmap overlay with the doctor-drawn box in white
idx = [i for i in range(len(ds)) if ds.df.iloc[i].Target == 1][:6]
fig, axs = plt.subplots(2, 6, figsize=(24, 8))
for c, i in enumerate(idx):
    x, y = ds[i]
    cam, prob = gradcam(model, x[None].to(dev))
    gray = (x[0] * CXRDataset.STD + CXRDataset.MEAN).numpy().clip(0, 1)
    axs[0, c].imshow(gray, cmap='gray'); axs[0, c].axis('off')
    axs[1, c].imshow(overlay(gray, cam)); axs[1, c].axis('off')
    axs[1, c].set_title(f'P(pneumonia)={prob:.2f}')
    for _, r in boxes[(boxes.patientId == ds.df.iloc[i].patientId) & (boxes.Target == 1)].iterrows():
        s = 224 / 1024  # boxes are in the original 1024px space
        axs[1, c].add_patch(patches.Rectangle((r.x*s, r.y*s), r.width*s, r.height*s, fill=False, edgecolor='white', lw=2))
plt.tight_layout(); plt.savefig('/content/drive/MyDrive/medai_ckpt/gradcam_examples.png', dpi=100); plt.show()